In [1]:
import os


In [2]:
%pwd

'd:\\pyhton\\Projects\\Kidny_disease\\kidney_Disease-classification-DL-MLflow\\research'

In [3]:
os.chdir('../')

In [4]:
%pwd

'd:\\pyhton\\Projects\\Kidny_disease\\kidney_Disease-classification-DL-MLflow'

In [5]:
import dagshub
dagshub.init(repo_owner='himanshusain0', repo_name='kidney_Disease-classification-DL-MLflow', mlflow=True)

import mlflow
with mlflow.start_run():
  mlflow.log_param('parameter name', 'value')
  mlflow.log_metric('metric name', 1)

Accessing as himanshusain0

Initialized MLflow to track repo "himanshusain0/kidney_Disease-classification-DL-MLflow"

Repository himanshusain0/kidney_Disease-classification-DL-MLflow initialized!

🏃 View run enthused-goat-445 at: https://dagshub.com/himanshusain0/kidney_Disease-classification-DL-MLflow.mlflow/#/experiments/0/runs/8b2de209b59440ceb8431fa6c576704e
🧪 View experiment at: https://dagshub.com/himanshusain0/kidney_Disease-classification-DL-MLflow.mlflow/#/experiments/0


In [6]:
import tensorflow as tf

In [8]:
model = tf.keras.models.load_model("artifacts/training/model.h5")

In [10]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int

In [11]:
from Kidney_disease.constants import *
from Kidney_disease.utils.common import read_yaml, create_directories, save_json

In [12]:
from dataclasses import dataclass
from pathlib import Path

In [13]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_evaluation_config(self) -> EvaluationConfig:

        eval_config = EvaluationConfig(
            path_of_model=Path("artifacts/training/model.h5"),
            training_data=Path("artifacts/data_ingestion/kidney-ct-scan-image"),
            mlflow_uri="https://dagshub.com/himanshusain0/kidney_Disease-classification-DL-MLflow.mlflow",
            all_params=self.params,
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
        )

        return eval_config

In [15]:

import tensorflow as tf
from pathlib import Path
import mlflow
import mlflow.keras
from urllib.parse import urlparse

In [8]:
print(EvaluationConfig)


NameError: name 'EvaluationConfig' is not defined

In [20]:
def get_evaluation_config(self) -> "EvaluationConfig":

SyntaxError: incomplete input (1488295430.py, line 1)

In [17]:
class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    
    def _valid_generator(self):

        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.30
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

    
    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)
    

    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = self.model.evaluate(self.valid_generator)
        self.save_score()

    def save_score(self):
        scores = {"loss": self.score[0], "accuracy": self.score[1]}
        save_json(path=Path("scores.json"), data=scores)

    
    def log_into_mlflow(self):
        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme
        
        with mlflow.start_run():
            mlflow.log_params(self.config.all_params)
            mlflow.log_metrics(
                {"loss": self.score[0], "accuracy": self.score[1]}
            )
            # Model registry does not work with file store
            if tracking_url_type_store != "file":

                # Register the model
                # There are other ways to use the Model Registry, which depends on the use case,
                # please refer to the doc for more information:
                # https://mlflow.org/docs/latest/model-registry.html#api-workflow
                mlflow.keras.log_model(self.model, "model", registered_model_name="VGG16Model")
            else:
                mlflow.keras.log_model(self.model, "model")



In [18]:

try:
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    evaluation = Evaluation(eval_config)
    evaluation.evaluation()
    evaluation.log_into_mlflow()

except Exception as e:
   raise e

Found 139 images belonging to 2 classes.
9/9 ━━━━━━━━━━━━━━━━━━━━ 34s 4s/step - accuracy: 0.9568 - loss: 0.1102


2026/06/16 16:58:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/16 16:58:44 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.
2026/06/16 16:59:22 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Registered model 'VGG16Model' already exists. Creating a new version of this model...
2026/06/16 17:00:07 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: VGG16Model, version 2
Created version '2' of model 'VGG16Model'.


🏃 View run dapper-crow-704 at: https://dagshub.com/himanshusain0/kidney_Disease-classification-DL-MLflow.mlflow/#/experiments/0/runs/2a12227229e44ed89b438e40298d07e1
🧪 View experiment at: https://dagshub.com/himanshusain0/kidney_Disease-classification-DL-MLflow.mlflow/#/experiments/0
